# 모형 학습

포함된 기상 입력과 학습 허용 지점의 노면온도로 모형을 학습한다. 전체 학습은 RUN_TRAINING 환경변수를 1로 지정한 뒤 실행한다. 기온·습도·풍속의 학습 명령은 weather/README.md를 따른다.

In [ ]:
from pathlib import Path
import json,os,sys,subprocess
import pandas as pd
root=Path.cwd().resolve()
if root.name=='notebooks':root=root.parent
plan=json.loads((root/'extended_training_plan.json').read_text(encoding='utf8'))
rows=[]
for role,record in plan['training_sources'].items():
 f=pd.read_parquet(root/record['file'])
 assert not set(f.station)&{'12410','23402','33411'}
 rows.append({'학습 목적':role,'행 수':len(f),'지점 수':f.station.nunique()})
display(pd.DataFrame(rows))

In [ ]:
if os.environ.get('RUN_TRAINING')=='1':
 target=root/'local_review'/'trained_model'
 env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',LOKY_MAX_CPU_COUNT='2')
 subprocess.run([sys.executable,'-X','utf8','code/train.py','--output',str(target)],cwd=root,env=env,check=True)
 display(json.loads((target/'refit_receipt.json').read_text(encoding='utf8')))
else:
 print('전체 학습 명령: python code/train.py --output local_review/trained_model')

## 도로 기상 학습

RUN_WEATHER_TRAINING을 1로 지정하면 기온·습도·풍속의 포함 학습 자료와 고정 설정으로 각각 학습한다.

In [ ]:
commands=[('air_temperature','retrain_weather_supplement.py'),('humidity','retrain_weather_history.py'),('wind','retrain_wind_network.py')]
if os.environ.get('RUN_WEATHER_TRAINING')=='1':
 env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',LOKY_MAX_CPU_COUNT='2')
 for name,script in commands:
  folder=root/'weather'/name
  subprocess.run([sys.executable,'-X','utf8',str(folder/'scripts'/script),'--bundle',str(folder),'--output',str(root/'local_review'/('trained_'+name))],cwd=folder,env=env,check=True)
else:
 display(pd.DataFrame(commands,columns=['기상 모형','학습 프로그램']))